# Colab T4 Claim Evidence Labeler

This notebook retrieves claim evidence from Qdrant, then labels the retrieved evidence with a local Gemma model on a Colab T4 GPU.

Expected input JSON can be raw claims or the same shape produced by `interactive_evidence_labeler.py`. Raw claims will be retrieved from Qdrant first; claims that already contain `retrieved_evidence`, `evidences`, `top_evidences`, or existing evidence fields can reuse those passages.

Set your `QDRANT_API_KEY` in Colab secrets or enter it when prompted.

In [ ]:
# Colab setup. Use Runtime -> Change runtime type -> T4 GPU before running.
!pip -q install -U transformers accelerate bitsandbytes sentencepiece qdrant-client sentence-transformers

In [ ]:
from google.colab import files
try:
    from google.colab import userdata
except Exception:
    userdata = None
from pathlib import Path
from datetime import datetime, timezone
import getpass
import json
import os
import re
import time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Upload Claims JSON

Upload either raw claims or a partial `*.llm_labeled.json` output. In resume mode, the notebook continues in the uploaded partial file instead of creating `*.llm_labeled.llm_labeled.json`.

In [ ]:
# Resume controls for partial labeling runs.
RESUME_MODE = True
RESUME_IN_PLACE = True

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

uploaded_paths = [Path(name) for name in uploaded.keys()]
resume_candidates = [
    path for path in uploaded_paths
    if ".llm_labeled" in path.stem or ".labeled" in path.stem
]
INPUT_PATH = resume_candidates[0] if RESUME_MODE and resume_candidates else uploaded_paths[0]

if RESUME_MODE and RESUME_IN_PLACE:
    OUTPUT_PATH = INPUT_PATH
elif INPUT_PATH.stem.endswith(".llm_labeled"):
    OUTPUT_PATH = INPUT_PATH
else:
    OUTPUT_PATH = INPUT_PATH.with_name(f"{INPUT_PATH.stem}.llm_labeled{INPUT_PATH.suffix}")

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)
print("Resume mode:", RESUME_MODE)

## Configure Model and Labeling

For a T4, use Gemma 4 E4B in 4-bit. 31B is too large for a normal T4 runtime.

In [ ]:
MODEL_ID = "google/gemma-4-E4B-it"
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
LLM_CLAIM_BATCH_SIZE = 1
MAX_EVIDENCE_CHARS = 1200

# Qdrant retrieval settings ported from interactive_evidence_labeler.py.
QDRANT_URL = os.getenv("QDRANT_URL", "<YOUR_QDRANT_URL>")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
if not QDRANT_API_KEY and userdata is not None:
    try:
        QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
    except Exception:
        QDRANT_API_KEY = None
if not QDRANT_API_KEY:
    QDRANT_API_KEY = getpass.getpass("QDRANT_API_KEY: ")

COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "medical_facts_global")
PRIMARY_MODEL = "BAAI/bge-m3"
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
CROSS_ENCODER_MODEL = "ncbi/MedCPT-Cross-Encoder"
USE_RERANKER = True
RETRIEVAL_DEVICE = "cpu"  # Keeps T4 VRAM available for Gemma. Use "cuda" only if you have headroom.

QUERY_POOL_SIZE = 200
RERANK_TOP_K = 40
TOP_K = 6
CLAIM_BATCH_SIZE = 5
QDRANT_WEIGHT = 0.35
CROSS_ENCODER_WEIGHT = 0.65
CROSS_ENCODER_BATCH_SIZE = 16
QDRANT_TIMEOUT = 90
FORCE_RETRIEVE = False
FORCE_RELABEL = False
INCLUDE_COMPLETE = False
RESUME_SKIP_ALREADY_LABELED = True

# Evidence fields this notebook will read from each claim.
EVIDENCE_FIELD_CANDIDATES = [
    "retrieved_evidence",
    "evidences",
    "top_evidences",
    "evidence",
    "positive_evidence",
    "negative_evidence",
]

In [ ]:
processor = None
tokenizer = None
model = None


def load_gemma_model():
    global processor, tokenizer, model
    if model is not None:
        return

    quant_config = None
    if LOAD_IN_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = processor.tokenizer
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        quantization_config=quant_config,
    )
    model.eval()
    print("Loaded", MODEL_ID)

In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def min_max_normalize(values):
    if not values:
        return []
    minimum = min(values)
    maximum = max(values)
    if maximum == minimum:
        return [0.0 for _ in values]
    return [(value - minimum) / (maximum - minimum) for value in values]


def batched(items, batch_size):
    for index in range(0, len(items), batch_size):
        yield items[index:index + batch_size]


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.write("\n")


def get_claims_ref(data):
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and isinstance(data.get("claims"), list):
        return data["claims"]
    raise ValueError("Input JSON must be a list or an object with a 'claims' list.")


def claim_text(claim):
    for key in ("claim", "text", "statement", "query"):
        value = claim.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    claim_id = claim.get("id", "<missing id>")
    raise ValueError(f"Claim {claim_id} has no text field. Expected claim/text/statement/query.")


def evidence_text(payload):
    parts = [payload.get("title") or "", payload.get("section_heading") or "", payload.get("text") or ""]
    return "\n".join(part for part in parts if part).strip()


def evidence_key(evidence):
    if evidence.get("evidence_id"):
        return str(evidence["evidence_id"])
    return "|".join(str(evidence.get(key) or "") for key in ("url", "title", "section_heading", "text"))


def unique_evidences(evidences):
    seen = set()
    out = []
    for evidence in evidences:
        if not isinstance(evidence, dict):
            continue
        key = evidence_key(evidence)
        if key in seen:
            continue
        seen.add(key)
        out.append(evidence)
    return out


def get_evidences(claim):
    gathered = []
    for field in EVIDENCE_FIELD_CANDIDATES:
        value = claim.get(field)
        if isinstance(value, list):
            gathered.extend(item for item in value if isinstance(item, dict))
        elif isinstance(value, dict):
            gathered.append(value)
    return unique_evidences(gathered)


def has_labeled_evidence(claim):
    for field in ("positive_evidence", "negative_evidence"):
        value = claim.get(field)
        if isinstance(value, list) and value:
            return True
    return False


def already_labeled(claim):
    if claim.get("labeling_complete"):
        return True
    if claim.get("llm_evidence_decisions"):
        return True
    return RESUME_SKIP_ALREADY_LABELED and has_labeled_evidence(claim)


def needs_retrieval(claim):
    if already_labeled(claim) and not INCLUDE_COMPLETE:
        return False
    return FORCE_RETRIEVE or not get_evidences(claim)


def needs_labeling(claim):
    if FORCE_RELABEL:
        return True
    if already_labeled(claim) and not INCLUDE_COMPLETE:
        return False
    return not already_labeled(claim)


def retrieval_metadata():
    return {
        "qdrant_url": QDRANT_URL,
        "collection_name": COLLECTION_NAME,
        "primary_model": PRIMARY_MODEL,
        "sapbert_model": SAPBERT_MODEL,
        "cross_encoder_model": CROSS_ENCODER_MODEL if USE_RERANKER else None,
        "query_pool_size": QUERY_POOL_SIZE,
        "rerank_top_k": RERANK_TOP_K,
        "top_k": TOP_K,
        "qdrant_weight": QDRANT_WEIGHT,
        "cross_encoder_weight": CROSS_ENCODER_WEIGHT,
        "retrieved_at": utc_now_iso(),
    }


def compact_evidence(row, rank):
    keys = (
        "rank", "evidence_id", "overall_score", "cross_score", "qdrant_score",
        "qdrant_score_norm", "cross_score_norm", "source", "url", "title",
        "section_heading", "text",
    )
    evidence = {key: row.get(key) for key in keys if row.get(key) is not None}
    evidence.setdefault("rank", rank)
    return evidence


def trim_text(value, max_chars):
    text = str(value or "").strip()
    if max_chars > 0 and len(text) > max_chars:
        return text[:max_chars - 3].rstrip() + "..."
    return text


class MedicalEvidenceRetriever:
    def __init__(self):
        from qdrant_client import QdrantClient, models as qdrant_models
        from sentence_transformers import SentenceTransformer
        from sentence_transformers.cross_encoder import CrossEncoder

        self.qdrant_models = qdrant_models
        self.collection_name = COLLECTION_NAME
        print(f"[Models] Loading primary encoder: {PRIMARY_MODEL} on {RETRIEVAL_DEVICE}")
        self.primary_model = SentenceTransformer(PRIMARY_MODEL, device=RETRIEVAL_DEVICE)
        print(f"[Models] Loading SapBERT encoder: {SAPBERT_MODEL} on {RETRIEVAL_DEVICE}")
        self.sapbert_model = SentenceTransformer(SAPBERT_MODEL, device=RETRIEVAL_DEVICE)
        self.cross_encoder = None
        if USE_RERANKER:
            print(f"[Models] Loading cross-encoder reranker: {CROSS_ENCODER_MODEL} on {RETRIEVAL_DEVICE}")
            self.cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512, device=RETRIEVAL_DEVICE)
        print(f"[Qdrant] Connecting to: {QDRANT_URL}")
        self.client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    def retrieve_batch(self, queries):
        if not queries:
            return []
        primary_vectors = self.primary_model.encode(queries, convert_to_numpy=True, normalize_embeddings=True).tolist()
        sapbert_vectors = self.sapbert_model.encode(queries, convert_to_numpy=True, normalize_embeddings=True).tolist()
        qdrant_results = self._query_qdrant_batch(primary_vectors, sapbert_vectors)
        rows_by_query = [self._rows_from_points(result.points[:RERANK_TOP_K]) for result in qdrant_results]
        if not USE_RERANKER or self.cross_encoder is None:
            return [self._top_qdrant_rows(rows) for rows in rows_by_query]
        return self._rerank_batch(queries, rows_by_query)

    def _query_qdrant_batch(self, primary_vectors, sapbert_vectors):
        requests = [
            self.qdrant_models.QueryRequest(
                prefetch=[
                    self.qdrant_models.Prefetch(query=vec_primary, using="primary", limit=QUERY_POOL_SIZE),
                    self.qdrant_models.Prefetch(query=vec_sapbert, using="sapbert", limit=QUERY_POOL_SIZE),
                ],
                query=self.qdrant_models.FusionQuery(fusion=self.qdrant_models.Fusion.RRF),
                limit=QUERY_POOL_SIZE,
                with_payload=True,
            )
            for vec_primary, vec_sapbert in zip(primary_vectors, sapbert_vectors)
        ]
        if hasattr(self.client, "query_batch_points"):
            return self.client.query_batch_points(collection_name=self.collection_name, requests=requests, timeout=QDRANT_TIMEOUT)
        return [
            self.client.query_points(
                collection_name=self.collection_name,
                prefetch=request.prefetch,
                query=request.query,
                limit=request.limit,
                with_payload=request.with_payload,
                timeout=QDRANT_TIMEOUT,
            )
            for request in requests
        ]

    def _rows_from_points(self, points):
        rows = []
        for qdrant_rank, point in enumerate(points, 1):
            payload = point.payload or {}
            text = evidence_text(payload)
            if not text:
                continue
            rows.append({
                "qdrant_rank": qdrant_rank,
                "evidence_id": str(point.id),
                "title": payload.get("title"),
                "section_heading": payload.get("section_heading"),
                "text": payload.get("text"),
                "qdrant_score": float(point.score),
                "source": payload.get("source_site"),
                "url": payload.get("url"),
            })
        return rows

    def _top_qdrant_rows(self, rows):
        for rank, row in enumerate(rows[:TOP_K], 1):
            row["rank"] = rank
            row["overall_score"] = row["qdrant_score"]
        return rows[:TOP_K]

    def _rerank_batch(self, queries, rows_by_query):
        pairs = []
        pair_refs = []
        for query_index, (query, rows) in enumerate(zip(queries, rows_by_query)):
            for row_index, row in enumerate(rows):
                text = evidence_text(row)
                if text:
                    pairs.append([query, text])
                    pair_refs.append((query_index, row_index))
        if not pairs:
            return [[] for _ in rows_by_query]
        cross_scores = self.cross_encoder.predict(pairs, batch_size=CROSS_ENCODER_BATCH_SIZE, show_progress_bar=False)
        for (query_index, row_index), cross_score in zip(pair_refs, cross_scores):
            rows_by_query[query_index][row_index]["cross_score"] = float(cross_score)
        reranked_by_query = []
        for rows in rows_by_query:
            scored_rows = [row for row in rows if row.get("cross_score") is not None]
            if not scored_rows:
                reranked_by_query.append([])
                continue
            qdrant_scores = [float(row["qdrant_score"]) for row in scored_rows]
            cross_scores_for_query = [float(row["cross_score"]) for row in scored_rows]
            for row, qdrant_norm, cross_norm in zip(scored_rows, min_max_normalize(qdrant_scores), min_max_normalize(cross_scores_for_query)):
                row["qdrant_score_norm"] = float(qdrant_norm)
                row["cross_score_norm"] = float(cross_norm)
                row["overall_score"] = (QDRANT_WEIGHT * row["qdrant_score_norm"]) + (CROSS_ENCODER_WEIGHT * row["cross_score_norm"])
            scored_rows.sort(key=lambda item: item["overall_score"], reverse=True)
            for rank, row in enumerate(scored_rows, 1):
                row["rank"] = rank
            reranked_by_query.append(scored_rows[:TOP_K])
        return reranked_by_query

In [ ]:
def build_prompt(claim_payloads):
    task = {
        "instructions": [
            "Label retrieved evidence for claim-evidence relevancy only.",
            "Do not decide whether the claim is true or false.",
            "Do not use entailment, support, contradiction, or refutation logic for the label.",
            "Use only the claim text and evidence text provided. Do not use outside medical knowledge.",
            "positive_evidence means the evidence is directly relevant and useful for judging the claim, whether it agrees with or disagrees with the claim.",
            "negative_evidence means the evidence is not relevant or not useful for judging the claim.",
            "neutral means partially related, background-only, ambiguous, too broad, or too weak.",
            "For false claims, evidence that refutes the claim is still positive_evidence if it is directly relevant.",
            "Return valid JSON only. Include one decision for every evidence_ref.",
        ],
        "output_schema": {
            "decisions": [
                {
                    "claim_ref": "string",
                    "evidence_ref": "string",
                    "label": "positive_evidence | negative_evidence | neutral",
                    "confidence": "number from 0 to 1",
                    "rationale": "short relevance reason, 25 words max",
                }
            ]
        },
        "claims": claim_payloads,
    }
    return json.dumps(task, ensure_ascii=False, indent=2)


def extract_json_object(text):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    if cleaned.startswith("{") and cleaned.endswith("}"):
        return cleaned
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        return cleaned[start:end + 1]
    return cleaned


def normalize_label(label):
    normalized = str(label or "").strip().lower().replace("-", "_").replace(" ", "_")
    if normalized in {"positive", "relevant", "directly_relevant", "useful", "positive_evidence"}:
        return "positive_evidence"
    if normalized in {"negative", "not_relevant", "irrelevant", "weak", "negative_evidence"}:
        return "negative_evidence"
    return "neutral"


def generate_json(prompt):
    load_gemma_model()
    messages = [
        {"role": "system", "content": "You are a strict JSON API. Return only valid JSON. Do not include markdown or prose."},
        {"role": "user", "content": prompt},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    if hasattr(encoded, "to"):
        encoded = encoded.to(model.device)
    elif torch.is_tensor(encoded):
        encoded = {"input_ids": encoded.to(model.device)}
    else:
        encoded = {key: value.to(model.device) if torch.is_tensor(value) else value for key, value in encoded.items()}
    prompt_token_count = encoded["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=TEMPERATURE > 0,
            temperature=TEMPERATURE if TEMPERATURE > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][prompt_token_count:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return json.loads(extract_json_object(text))

In [ ]:
def build_claim_payload(claim_ref, claim, evidences):
    evidence_payloads = []
    for idx, evidence in enumerate(evidences, 1):
        evidence_payloads.append({
            "evidence_ref": f"e{idx}",
            "rank": evidence.get("rank", idx),
            "overall_score": evidence.get("overall_score"),
            "cross_score": evidence.get("cross_score"),
            "qdrant_score": evidence.get("qdrant_score"),
            "source": evidence.get("source"),
            "url": evidence.get("url"),
            "title": trim_text(evidence.get("title"), 300),
            "section_heading": trim_text(evidence.get("section_heading"), 300),
            "text": trim_text(evidence.get("text"), MAX_EVIDENCE_CHARS),
        })
    return {
        "claim_ref": claim_ref,
        "claim": claim_text(claim),
        "evidences": evidence_payloads,
    }


def label_claim_batch(batch):
    claim_payloads = []
    evidence_by_claim_ref = {}
    item_by_claim_ref = {}

    for claim_index, claim in batch:
        evidences = get_evidences(claim)
        claim_ref = f"claim_{claim_index + 1}"
        evidence_by_claim_ref[claim_ref] = evidences
        item_by_claim_ref[claim_ref] = claim
        if evidences:
            claim_payloads.append(build_claim_payload(claim_ref, claim, evidences))

    if not claim_payloads:
        for _claim_index, claim in batch:
            claim.setdefault("positive_evidence", [])
            claim.setdefault("negative_evidence", [])
            claim["llm_evidence_decisions"] = []
            claim["labeling_complete"] = False
            claim["labeling_error"] = "No evidence retrieved for this claim."
        return

    response = generate_json(build_prompt(claim_payloads))
    decisions = response.get("decisions", [])
    if not isinstance(decisions, list):
        raise RuntimeError("Model response has no decisions list.")

    decisions_by_claim = {}
    for decision in decisions:
        if isinstance(decision, dict):
            decisions_by_claim.setdefault(decision.get("claim_ref"), {})[decision.get("evidence_ref")] = decision

    for claim_ref, claim in item_by_claim_ref.items():
        evidences = evidence_by_claim_ref[claim_ref]
        if not evidences:
            claim.setdefault("positive_evidence", [])
            claim.setdefault("negative_evidence", [])
            claim["llm_evidence_decisions"] = []
            claim["labeling_complete"] = False
            claim["labeling_error"] = "No evidence retrieved for this claim."
            continue
        positive = []
        negative = []
        decision_log = []
        claim_decisions = decisions_by_claim.get(claim_ref, {})

        for idx, row in enumerate(evidences, 1):
            evidence_ref = f"e{idx}"
            decision = claim_decisions.get(evidence_ref, {})
            label = normalize_label(decision.get("label"))
            evidence = compact_evidence(row, idx)
            evidence["llm_label"] = label
            evidence["llm_confidence"] = decision.get("confidence")
            evidence["llm_rationale"] = decision.get("rationale")
            evidence["llm_model"] = MODEL_ID

            decision_log.append({
                "evidence_ref": evidence_ref,
                "evidence_id": evidence.get("evidence_id"),
                "rank": evidence.get("rank"),
                "label": label,
                "confidence": decision.get("confidence"),
                "rationale": decision.get("rationale"),
            })

            if label == "positive_evidence":
                positive.append(evidence)
            elif label == "negative_evidence":
                negative.append(evidence)

        claim["positive_evidence"] = positive
        claim["negative_evidence"] = negative
        claim["llm_evidence_decisions"] = decision_log
        claim["labeling_complete"] = True
        claim["labeling_completed_by"] = MODEL_ID
        claim["labeling_completed_at"] = utc_now_iso()
        claim.pop("labeling_error", None)

In [ ]:
data = load_json(INPUT_PATH)
claims = get_claims_ref(data)
print("Claims:", len(claims))
print("Already labeled claims:", sum(1 for claim in claims if already_labeled(claim)))
print("Claims with saved evidence:", sum(1 for claim in claims if get_evidences(claim)))

claims_needing_retrieval = [(idx, claim) for idx, claim in enumerate(claims) if needs_retrieval(claim)]
print("Claims needing retrieval:", len(claims_needing_retrieval))

if claims_needing_retrieval:
    retriever = MedicalEvidenceRetriever()
    for batch_number, batch in enumerate(batched(claims_needing_retrieval, CLAIM_BATCH_SIZE), 1):
        query_items = [(idx, claim, claim_text(claim)) for idx, claim in batch]
        print(f"Retrieving batch {batch_number}: claims {[idx + 1 for idx, _, _ in query_items]}")
        retrieved_by_claim = retriever.retrieve_batch([text for _, _, text in query_items])
        for (idx, claim, _text), evidences in zip(query_items, retrieved_by_claim):
            claim["retrieved_evidence"] = [compact_evidence(row, rank + 1) for rank, row in enumerate(evidences)]
            claim["retrieval_metadata"] = retrieval_metadata()
            claim["labeling_complete"] = False
            if evidences:
                claim.pop("labeling_error", None)
            else:
                claim["labeling_error"] = "Qdrant returned no evidence for this claim."
            print(f"Claim {idx + 1}: retrieved {len(evidences)} evidence item(s)")
        save_json(OUTPUT_PATH, data)
    del retriever
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pending = [(idx, claim) for idx, claim in enumerate(claims) if needs_labeling(claim)]
print("Claims pending Gemma labeling:", len(pending))

for start in range(0, len(pending), LLM_CLAIM_BATCH_SIZE):
    batch = pending[start:start + LLM_CLAIM_BATCH_SIZE]
    print(f"Labeling batch {start // LLM_CLAIM_BATCH_SIZE + 1}: claims {[idx + 1 for idx, _ in batch]}")
    try:
        label_claim_batch(batch)
    except Exception as exc:
        print("Batch failed:", exc)
        if LLM_CLAIM_BATCH_SIZE != 1:
            print("Retrying individually...")
            for item in batch:
                label_claim_batch([item])
        else:
            raise
    save_json(OUTPUT_PATH, data)

print("Saved:", OUTPUT_PATH)

In [ ]:
files.download(str(OUTPUT_PATH))